# Surface Crack Detection - Training
This notebook is designed to be cross-compatible with Kaggle, Google Colab, and local environments.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

print(f'PyTorch version: {torch.__version__}')

In [ ]:
# ==========================================
# CONFIGURATION - ADJUST THESE PATHS
# ==========================================

# Default path assumes local or Kaggle structure.
# If on Kaggle, the dataset is usually in /kaggle/input/...
DATASET_PATH = '/kaggle/input/datasets/yidazhang07/bridge-cracks-image'

# Adjust these hyper-parameters as needed
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
MODEL_SAVE_PATH = 'model_weights.pth'

# Environment Detection
if 'KAGGLE_URL_BASE' in os.environ:
    print('Running on Kaggle')
elif 'COLAB_GPU' in os.environ:
    print('Running on Google Colab')
    # If on Colab, you might need to mount drive or download dataset
    # DATASET_PATH = '/content/dataset'
else:
    print('Running Locally')
    # Override local path if testing on laptop
    # DATASET_PATH = '../data/' 


In [ ]:
# Data Augmentation & Loading
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Assuming dataset is organized in folders by class
try:
    dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform_train)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    # Override transform for validation set
    val_dataset.dataset.transform = transform_test

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    num_classes = len(dataset.classes)
    print(f'Classes ({num_classes}): {dataset.classes}')
except Exception as e:
    print(f'Error loading dataset from {DATASET_PATH}. Please check the path.')
    print(f'Exception: {e}')
    num_classes = 4 # Fallback for model definition


In [ ]:
# Model Definition (Transfer Learning with MobileNetV2)
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze early layers if desired (optional)
# for param in model.parameters():
#     param.requires_grad = False

# Replace classifier head
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [ ]:
# Training Loop
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Skip training loop if dataset wasn't loaded properly in the previous cell
    if 'train_loader' not in locals():
        print('Dataset not loaded. Skipping training loop.')
        break

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    train_acc = 100. * correct / total
    
    # Validation
    model.eval()
    val_loss = 0.0
    v_correct = 0
    v_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            v_total += labels.size(0)
            v_correct += predicted.eq(labels).sum().item()
            
    val_acc = 100. * v_correct / v_total
    print(f'Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/len(train_loader):.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss/len(val_loader):.4f} Acc: {val_acc:.2f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f'Saved best model with acc: {best_acc:.2f}%')


In [ ]:
# Download Weights locally
import os

if os.path.exists(MODEL_SAVE_PATH):
    print(f'Model saved at {MODEL_SAVE_PATH}')
    if 'KAGGLE_URL_BASE' in os.environ:
        from IPython.display import FileLink
        display(FileLink(MODEL_SAVE_PATH))
        print('Click the link above to download the weights.')
    elif 'COLAB_GPU' in os.environ:
        from google.colab import files
        files.download(MODEL_SAVE_PATH)
        print('Downloading weights...')
    else:
        print('Weights are saved locally in the current directory.')
else:
    print('Model weights file not found. Did the model train successfully?')
